# Imports and Setup

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("""use catalog novocart_catalog""")
spark.sql("""create schema if not exists gold_schema""")

gold_run_id = str(uuid.uuid4())
run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print(f"Current Run Id: {gold_run_id}")
print(f"Current ts folder: {run_ts_str}")

# Gold Control Table

In [0]:
spark.sql("""
          create table if not exists novocart_catalog.gold_schema.processing_control (
              layer string,
              entity_name string,
              last_processed_silver_run_id string,
              last_processed_silver_run_ts timestamp,
              rows_merged bigint,
              run_status string,
              gold_run_id string,
              updated_at timestamp
          ) 
          using delta
          """)

# Helper Functions

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)

        (
            dt.alias("t")
            .merge(df_source.alias("s"), f"t.{join_key} = s.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name: str):
    ctrl = (
        spark.read.table("novocart_catalog.gold_schema.processing_control")
        .filter(
            (col("layer") == "gold")
            & (col("entity_name") == entity_name)
            & (col("run_status") == "success")
        )
        .orderBy(col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None
    else:
        return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(
    entity_name, last_processed_silver_run_id, last_processed_silver_ts, rows_merged
):
    ctrl_df = spark.createDataFrame(
        [
            (
                "gold",
                entity_name,
                last_processed_silver_run_id,
                last_processed_silver_ts,
                rows_merged,
                "success",
                gold_run_id,
                datetime.utcnow(),
            )
        ],
        schema="layer string, entity_name string, last_processed_silver_run_id string, last_processed_silver_run_ts timestamp, rows_merged bigint, run_status string, gold_run_id string, updated_at timestamp",
    )

    dt = DeltaTable.forName(spark, "novocart_catalog.gold_schema.processing_control")

    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(
            set={
                "last_processed_silver_run_id": "s.last_processed_silver_run_id",
                "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
                "rows_merged": "s.rows_merged",
                "run_status": "s.run_status",
                "gold_run_id": "s.gold_run_id",
                "updated_at": "s.updated_at",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

# Read Changed Silver Rows Only

In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")

print(f"Last Processed Silver Timestamp for gold: {last_gold_ts}")

silver_orders_current = spark.read.table("novocart_catalog.silver_schema.orders_transformed")
silver_products_current =  spark.read.table("novocart_catalog.silver_schema.products_transformed")
silver_payments_current =  spark.read.table("novocart_catalog.silver_schema.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    changed_orders = silver_orders_current.filter(col("updated_at") > lit(last_gold_ts))
    changed_products = silver_products_current.filter(col("updated_at") > lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(col("updated_at") > lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed orders: {changed_orders_count}")
print(f"Number of changed products: {changed_products_count}")
print(f"Number of changed payments: {changed_payments_count}")

# Impacted Order ID

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
imapcted_from_products = (
    changed_products.alias("p")
    .join(
        silver_orders_current.alias("o"),
        col("p.product_id") == col("o.product_id"),
        "inner",
    )
    .select("o.order_id")
    .distinct()
)

impacted_order_id = (
    impacted_from_orders.union(impacted_from_payments)
    .union(imapcted_from_products)
    .distinct()
)

print("impacted order ids: ", impacted_order_id.count())
display(impacted_order_id.orderBy("order_id"))

In [0]:
impacted_orders = silver_orders_current.join(
    impacted_order_id, ["order_id"], "inner"
)

gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        col("o.product_id") == col("p.product_id"),
        "inner",
    )
    .join(
        silver_payments_current.alias("pay"),
        col("o.order_id") == col("pay.order_id"),
        "inner",
    )
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("p.product_id"),
        col("p.product_name"),
        col("p.category"),
        col("p.price").alias("product_price"),
        col("o.order_status"),
        col("o.order_amount"),
        col("pay.payment_id"),
        col("pay.payment_status"),
        col("pay.paid_amount"),
        col("o.order_date"),
        col("o.order_month"),
        col("o.order_year"),
        greatest(
            col("o.updated_at").cast("timestamp"),
            col("p.updated_at").cast("timestamp"),
            col("pay.processed_at").cast("timestamp"),
        ).alias("gold_update_ts"),
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        when(
            col("order_amount") > 0, round( col("paid_amount") / col("order_amount"), 2)
        ).otherwise(lit(0.0)),
    )
    .withColumn(
        "payment_state",
        when(col("order_amount") == 0, "invalid_order_amount")
        .when(col("payment_completion_ratio") == 0, "Unpaid")
        .when(col("payment_completion_ratio") == 1, "Paid")
        .when(col("payment_completion_ratio") < 1, "Partially_paid")
        .when(col("payment_completion_ratio") > 1, "Over_paid"),
    )
    .withColumn("gold_updated_date", to_date(col("gold_update_ts")))
    .withColumn("gold_run_id", lit(gold_run_id))
)

print("gold_delta_rows = ", gold_delta.count())
display(gold_delta)

# Merge gold Current-State Table

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "novocart_catalog.gold_schema.orders_information", "order_id")
else:
    print("No new record to upsert in gold table")

# Maintain Gold SCD Type 2 history

In [0]:
if not spark.catalog.tableExists(
    "novocart_catalog.gold_schema.orders_information_scd2"
):
    spark.sql("""
              create table novocart_catalog.gold_schema.orders_information_scd2
              using delta
              as 
              select *,
              cast(null as timestamp) as valid_from_ts,
              cast(null as timestamp) as valid_to_ts,
              true as is_current
              from novocart_catalog.gold_schema.orders_information
              where 1= 0  
              """)
    
if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
              merge into novocart_catalog.gold_schema.orders_information_scd2 as t
              using gold_delta_view as s
              on t.order_id = s.order_id
              and t.is_current = true 
              when matched and (
                  not(t.order_status <=> s.order_status) or 
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price) )
                  then update set
                  is_current = false,
                  valid_to_ts = s.gold_update_ts
              """)
    
    spark.sql("""
              insert into novocart_catalog.gold_schema.orders_information_scd2
              select s.*,
              s.gold_update_ts as valid_from_ts,
              cast(null as timestamp) as valid_to_ts,
              true as is_current
              from gold_delta_view as s
              left join novocart_catalog.gold_schema.orders_information_scd2 t 
              on s.order_id = t.order_id and is_current = true
              where t.order_id is null or (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price)
              )
              """)

In [0]:
%sql
select * from novocart_catalog.gold_schema.orders_information_scd2

# Update category-level Gold Aggregation

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta.select("category").filter(col("category").isNotNull()).distinct()
    )

    category_perf_delta = (
        spark.read.table("novocart_catalog.gold_schema.orders_information")
        .join(impacted_categories, ["category"], "inner")
        .groupBy("category")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            sum(
                when(col("order_amount") > 0, col("order_amount"))
                .otherwise(0.0)
            ).alias("Gross_merchendise_value"),
            sum(
                when(col("paid_amount") > 0, col("paid_amount"))
                .otherwise(0.0)
            ).alias("Total_Paid_Amount"),
            avg(col("payment_completion_ratio")).alias("Avg_Payment_Completion_Ratio"),
            (sum(when(col("payment_state") == "Failed", 1).otherwise(0))/count("*")).alias("Payment_Failure_Rate")
        )
    )

    upsert_to_gold(category_perf_delta, "novocart_catalog.gold_schema.category_performance", "category")
else:
    print("No new record to upsert in gold table")

In [0]:
%sql
select * from novocart_catalog.gold_schema.category_performance

# Publish Gold Snapshot to Volume

In [0]:
spark.sql("""
          create volume if not exists novocart_catalog.gold_schema.gold_snapshot_vol
          """)

In [0]:
latest_orders_path = "/Volumes/novocart_catalog/gold_schema/gold_snapshot_vol/gold_latest/orders_information"

latest_category_path = "/Volumes/novocart_catalog/gold_schema/gold_snapshot_vol/gold_latest/category_performance"

historical_orders_path = f"/Volumes/novocart_catalog/gold_schema/gold_snapshot_vol/gold_snapshots/orders_information/rundate = {run_date_str}/run_ts = {run_ts_str}"

historical_category_path = f"/Volumes/novocart_catalog/gold_schema/gold_snapshot_vol/gold_snapshots/category_performance/rundate = {run_date_str}/run_ts = {run_ts_str}"

spark.read.table("novocart_catalog.gold_schema.orders_information").write.format("parquet").mode("overwrite").save(latest_orders_path)

spark.read.table("novocart_catalog.gold_schema.category_performance").write.format("parquet").mode("overwrite").save(latest_category_path)

spark.read.table("novocart_catalog.gold_schema.orders_information").write.format("parquet").mode("overwrite").save(historical_orders_path)

spark.read.table("novocart_catalog.gold_schema.category_performance").write.format("parquet").mode("overwrite").save(historical_category_path)

print(f"Latest orders path: ", latest_orders_path)
print(f"Latest category path: ", latest_category_path)
print(f"Historical orders path: ", historical_orders_path)
print(f"Historical category path: ", historical_category_path)


# Update Gold Control Table

In [0]:
latest_silver_ts = silver_orders_current.agg(
    max("bronze_ingested_at").alias("mx")
).collect()[0]["mx"]

latest_silver_run_id = (
    (
        silver_orders_current.filter(col("bronze_ingested_at") == latest_silver_ts)
        .agg(max("silver_run_id").alias("mx"))
        .collect()[0]["mx"]
    )
    if latest_silver_ts is not None
    else None
)

upsert_gold_control(
    "orders_information", latest_silver_run_id, latest_silver_ts, gold_delta.count()
)
display(spark.read.table("novocart_catalog.gold_schema.processing_control"))